In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

article_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/DataMining/Do_an/Crawl_data/vov/vov_coffee_articles.csv')
output_path = "/content/drive/My Drive/Colab Notebooks/DataMining/Do_an/Crawl_data/vov/vov_coffee_prices.csv"

Mounted at /content/drive


In [2]:
import os
import json
import re
import time
from tqdm import tqdm
import gc
import csv

In [ ]:
# pd.set_option('display.max_colwidth', None)
article_df.head()

In [4]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00


In [5]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Assuming HF_TOKEN is set as a Colab secret
login(token=HF_TOKEN)

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [7]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [8]:
def build_prompt(text):

    text = text.replace("\n", " ")

    prompt = f"""
You are an information extraction system.

Extract coffee price information from the Vietnamese article.

Return ONLY valid JSON.

Do not explain.
Do not write any text before or after JSON.

Output must follow EXACTLY this schema:

{{
  "price_date": "DD/MM",
  "domestic_market": {{
    "average_price_vnd_per_kg": number,
    "province_prices": [
      {{"province": "dak_lak", "price": number}},
      {{"province": "dak_nong", "price": number}},
      {{"province": "lam_dong", "price": number}},
      {{"province": "gia_lai", "price": number}},
      {{"province": "kon_tum", "price": number}}
    ]
  }},
  "global_market": {{
    "robusta_price_usd_per_ton": number,
    "arabica_price_cent_per_lb": number
  }},
  "trend": {{
    "domestic": "up|down|stable",
    "global": "up|down|stable"
  }}
}}

Constraints:
- Output must start with {{
- Output must end with }}
- Only JSON is allowed
- All numbers must be integers
- Province names must be exactly:
  dak_lak, dak_nong, lam_dong, gia_lai, kon_tum

Article:
{text}
"""

    return prompt

In [9]:
def extract_information(text):
  prompt = build_prompt(text)
  inputs = tokenizer(prompt, return_tensors="pt")
  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  outputs = model.generate( **inputs, max_new_tokens=300, do_sample=False ) # chỉ lấy phần model sinh ra

  generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
  result = tokenizer.decode( generated_tokens, skip_special_tokens=True )

  return result

In [10]:
def extract_first_json(text):

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        return None

    json_text = text[start:end+1]

    return json_text

In [11]:
def safe_parse_json(llm_output):

    try:

        json_text = extract_first_json(llm_output)

        if json_text is None:
            return None

        data = json.loads(json_text)

        return data

    except Exception:
        return None

In [12]:
def test_extract_article(row):

    text = row["text"]

    llm_output = extract_information(text)

    data = safe_parse_json(llm_output)

    if data is None:
        print("JSON parse failed")
        return None

    try:

        avg_price = data["domestic_market"]["average_price_vnd_per_kg"]
        robusta = data["global_market"]["robusta_price_usd_per_ton"]
        arabica = data["global_market"]["arabica_price_cent_per_lb"]

        trend_domestic = data["trend"]["domestic"]
        trend_global = data["trend"]["global"]

        price_date = data["price_date"]

    except:
        print("Missing key in JSON")
        return None


    province_dict = {
        "dak_lak": None,
        "dak_nong": None,
        "lam_dong": None,
        "gia_lai": None,
        "kon_tum": None
    }

    province_prices = data.get("domestic_market", {}).get("province_prices", [])

    for p in province_prices:

        province = p.get("province", "").lower().replace(" ", "_")

        price = p.get("price") or p.get("price_vnd_per_kg")

        if province in province_dict:
            province_dict[province] = price


    record = {
        "url": row["url"],
        "title": row["title"],
        "publish_date": row["publish_date"],
        "price_date": price_date,

        "average_price": avg_price,
        "robusta_price": robusta,
        "arabica_price": arabica,

        "trend_domestic": trend_domestic,
        "trend_global": trend_global,

        "llm_output": llm_output.replace("\n"," ").replace("\r"," "),
        "raw": text.replace("\n"," ").replace("\r"," ")
    }

    record.update(province_dict)

    df = pd.DataFrame([record])

    return df

In [13]:
def run_all_articles(article_df, output_path):

    if os.path.exists(output_path):

        print("Resume from existing file")

        old_df = pd.read_csv(output_path)

        processed_urls = set(old_df["url"])

        article_df = article_df[~article_df["url"].isin(processed_urls)]

        write_header = False

    else:

        print("Start new file")

        write_header = True


    print("Articles left:", len(article_df))


    for i in tqdm(range(len(article_df))):

        row = article_df.iloc[i]

        try:

            result_df = test_extract_article(row)

            if result_df is None:
                continue

            result_df.to_csv(
                output_path,
                mode="a",
                header=write_header,
                index=False,
                quoting=csv.QUOTE_ALL
            )

            write_header = False

        except Exception as e:

            print("Error:", e)
            continue

    print("Finished!")

In [14]:
run_all_articles(article_df, output_path)

Start new file
Articles left: 956


  7%|▋         | 63/956 [13:14<3:02:01, 12.23s/it]

JSON parse failed


 10%|▉         | 92/956 [19:18<2:15:43,  9.43s/it]

Missing key in JSON


...

JSON parse failed


 99%|█████████▊| 944/956 [3:21:20<02:26, 12.19s/it]

JSON parse failed


100%|██████████| 956/956 [3:23:36<00:00, 12.78s/it]

JSON parse failed
Error: 'float' object has no attribute 'replace'
Finished!
